# Foundations · The harness

A model gives a different answer every time you ask. So "it worked when I tried it" proves little. A **harness** asks the same questions every time, records every request and response, and turns them into a score. It is to an AI system what a test suite is to ordinary code, and it is the only honest way to say "this change made things better".

Ours has four parts:

| Part | File | Role |
|---|---|---|
| Client | `harness/client.py` | Sends HTTP requests, retries on 429/503, records every exchange as evidence |
| Checks | `harness/checks/week*.py` | Functions that return a score from 0 to 1 and a detail line |
| Data | `harness/data/week*.jsonl` | The public questions and expected answers |
| Engine and report | `harness/engine.py`, `harness/report.py` | Weighted score out of 100; a Markdown report and a JSON proof with a SHA-256 digest |

Grading runs the same engine with more, unseen data. That is why you should improve the *capability*, not tune for the public questions.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()
course.run([sys.executable, "-m", "harness", "list"])

## 1 · A check is a small function

In [ ]:
import inspect

from harness.checks import week05

print(inspect.getsource(week05._accuracy))

## 2 · Run it without Azure

`in_process=True` runs the service inside this notebook. Setting `LLM_MODE=offline` swaps the models for deterministic stand-ins, so the run costs nothing and takes seconds. The offline score is meaningless as a grade; the point here is the mechanics.

In [ ]:
import os

os.environ["LLM_MODE"] = "offline"
try:
    proof = course.check(5, in_process=True)
finally:
    os.environ["LLM_MODE"] = "foundry"

## 3 · Read the proof

Each check keeps the exchanges it made. When a check fails, this is where you find out why: the exact request, the exact response, the status and the latency.

In [ ]:
import json

print("score: ", proof["score"])
print("digest:", proof["digest"])
check = next(c for c in proof["checks"] if c["id"] == "w05.validation")
for exchange in check["exchanges"]:
    print(exchange["status"], exchange["method"], exchange["path"], json.dumps(exchange["request"])[:60])

## 4 · Write your own check

A check takes a `Context` (with `ctx.target` to call the service) and returns `(score, detail)`. This one verifies that the service reports a real version, not `dev`.

In [ ]:
from fastapi.testclient import TestClient

from app.config import LLMMode, Settings
from app.main import create_app
from harness.client import Target
from harness.engine import Context


def version_is_set(ctx: Context) -> tuple[float, str]:
    info = ctx.target.get("/v1/info").json()
    ok = info.get("version") not in (None, "", "dev")
    return (1.0 if ok else 0.0), f"version = {info.get('version')!r}"


settings = Settings(llm_mode=LLMMode.OFFLINE, student_id="demo", app_version="dev", _env_file=None)
with TestClient(create_app(settings)) as client:
    print(version_is_set(Context(target=Target(client=client), week=5)))

## What to remember

- Measure before and after every change. One score, same data, same checks.
- Read failures in the proof, not in your head: request, response, status, latency.
- Public data is for learning; the grade uses unseen data. Generalize.
- A check is a function. When you find a failure mode, write a check for it.

**Learn more:** [Effective harnesses for long-running agents (Anthropic)](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) · [Operationalize generative AI applications (Learn path)](https://learn.microsoft.com/en-us/training/paths/operationalize-gen-ai-apps/)